# Panerai Deep Analysis Dashboard
This notebook provides advanced business intelligence on Panerai watch distribution, pricing extremes, and market availability across different countries and collections.

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
from pyspark.sql import functions as F

# Environment Configuration
try:
    env = dbutils.widgets.get("env")
except:
    env = "prod"

print(f"🚀 Running deep analysis in environment: {env}")

# Base catalog pattern
CATALOG = "panerai_project"
SCHEMA = env

## 1. Data Layer Initialization
We create a unified temporary view joining the Gold layer tables. We specifically handle the date format 'dd-MM-yyyy' to ensure correct chronological analysis.

In [ ]:
tables = {
    "collection": f"{CATALOG}.{SCHEMA}.dim_collection",
    "country": f"{CATALOG}.{SCHEMA}.dim_country",
    "watch": f"{CATALOG}.{SCHEMA}.dim_watch",
    "pricing": f"{CATALOG}.{SCHEMA}.fact_pricing"
}

spark.sql(f"""
    CREATE OR REPLACE TEMPORARY VIEW v_deep_analysis AS
    SELECT 
        p.date, p.price_eur, p.diameter_mm, p.reference, p.url,
        c.collection, 
        cnt.country, cnt.currency,
        w.title, w.edition
    FROM {tables['pricing']} p
    JOIN {tables['collection']} c ON p.collection_id = c.collection_id
    JOIN {tables['country']} cnt ON p.country_id = cnt.country_id
    JOIN {tables['watch']} w ON p.watch_id = w.watch_id
""")

print("✅ Unified Deep Analysis View created.")

## 2. High-Level KPIs
Quick summary of the current market state.

In [ ]:
# Calculate Median and Total Offer
# Spark SQL does not have a simple MEDIAN() function, so we use percentile_approx(col, 0.5)
kpi_df = spark.sql(f"""
    SELECT 
        COUNT(*) as total_offer, 
        percentile_approx(price_eur, 0.5) as median_price 
    FROM v_deep_analysis
""").toPandas()

print(f"Total Offers: {int(kpi_df['total_offer'][0])}")
print(f"Global Median Price: {kpi_df['median_price'][0]:.2f} EUR")

# New offers per country (most recent date)
recent_date = spark.sql("SELECT MAX(date) FROM v_deep_analysis").collect()[0][0]
current_market_df = spark.sql(f"""
    SELECT country, COUNT(*) as count 
    FROM v_deep_analysis 
    WHERE date = '{recent_date}'
    GROUP BY country
""").toPandas()

print(f"\\nOffers per country on {recent_date}:")
print(current_market_df.to_string(index=False))

## 3. Distribution Trends
Analysis of how the number of watches evolves per country and collection.

In [ ]:
# Watches per Country over time
country_trend_df = spark.sql("""
    SELECT date, country, COUNT(reference) as count 
    FROM v_deep_analysis 
    GROUP BY date, country 
    ORDER BY date
""").toPandas()

fig_country_trend = px.line(
    country_trend_df, x='date', y='count', color='country', 
    title='Watch Distribution Trend per Country', template='plotly_white'
)
fig_country_trend.show()

# Watches per Collection over time
coll_trend_df = spark.sql("""
    SELECT date, collection, COUNT(reference) as count 
    FROM v_deep_analysis 
    GROUP BY date, collection 
    ORDER BY date
""").toPandas()

fig_coll_trend = px.line(
    coll_trend_df, x='date', y='count', color='collection', 
    title='Watch Distribution Trend per Collection', template='plotly_white'
)
fig_coll_trend.show()

## 4. Median Price Distribution
Visualizing the median value share per segment.

In [ ]:
# Median Price per Country
country_price_df = spark.sql("""
    SELECT country, percentile_approx(price_eur, 0.5) as median_price 
    FROM v_deep_analysis 
    GROUP BY country
""").toPandas()

fig_country_pie = px.pie(
    country_price_df, values='median_price', names='country', 
    title='Median Price Distribution per Country', template='plotly_white'
)
fig_country_pie.show()

# Median Price per Collection
coll_price_df = spark.sql("""
    SELECT collection, percentile_approx(price_eur, 0.5) as median_price 
    FROM v_deep_analysis 
    GROUP BY collection
""").toPandas()

fig_coll_pie = px.pie(
    coll_price_df, values='median_price', names='collection', 
    title='Median Price Distribution per Collection', template='plotly_white'
)
fig_coll_pie.show()

## 5. Price Extremes
Identifying the most and least expensive models and market variances.

In [ ]:
# Most and Least Expensive Watch per Collection
extremes_sql = """
WITH CollectionRankings AS (
    SELECT 
        collection, title, reference, price_eur, country,
        ROW_NUMBER() OVER (PARTITION BY collection ORDER BY price_eur DESC) as rank_desc,
        ROW_NUMBER() OVER (PARTITION BY collection ORDER BY price_eur ASC) as rank_asc, url
    FROM v_deep_analysis
)
SELECT reference, title, collection, country, price_eur, 'Most Expensive' as label, url
FROM CollectionRankings WHERE rank_desc = 1
UNION ALL
SELECT reference, title, collection, country, price_eur, 'Least Expensive' as label, url
FROM CollectionRankings WHERE rank_asc = 1
ORDER BY collection
"""
extremes_df = spark.sql(extremes_sql).toPandas()
display(extremes_df)

# Market Variance per Reference (Cheapest vs Most Expensive Country)
variance_sql = """
WITH ReferenceExtremes AS (
    SELECT 
        reference,
        country,
        price_eur,
        url,
        FIRST_VALUE(country) OVER (PARTITION BY reference ORDER BY price_eur ASC) as cheapest_country,
        FIRST_VALUE(price_eur) OVER (PARTITION BY reference ORDER BY price_eur ASC) as min_price,
        FIRST_VALUE(url) OVER (PARTITION BY reference ORDER BY price_eur ASC) as min_price_url,
        FIRST_VALUE(country) OVER (PARTITION BY reference ORDER BY price_eur DESC) as most_expensive_country,
        FIRST_VALUE(price_eur) OVER (PARTITION BY reference ORDER BY price_eur DESC) as max_price,
        FIRST_VALUE(url) OVER (PARTITION BY reference ORDER BY price_eur DESC) as max_price_url
    FROM v_deep_analysis
)
SELECT 
    reference,
    ROUND((max_price - min_price) / min_price * 100, 2)      AS arbitrage_pct,
    ROUND(max_price - min_price, 2)                          AS arbitrage_delta_eur,
    cheapest_country,
    min_price,
    min_price_url,
    most_expensive_country,
    max_price,
    max_price_url
FROM ReferenceExtremes
GROUP BY 1, 2, 3, 4, 5, 6, 7, 8, 9
ORDER BY arbitrage_pct DESC, reference
"""
variance_df = spark.sql(variance_sql).toPandas()
display(variance_df)